In [ ]:
!pip -q install -U chronos-forecasting wandb pyarrow
import os, numpy as np, pandas as pd, torch
print("CUDA:", torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")
import wandb; wandb.login()
os.environ.setdefault("WANDB_PROJECT", "chronos-taxi-ft")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
DATA_PATH   = "/content/drive/MyDrive/Timesfm_Chronos_fine-tune/data/taxi_series.parquet"
CKPT_DIR    = "/content/drive/MyDrive/Timesfm_Chronos_fine-tune/chronos_ft"
RESOLUTIONS = []      # test on 1h first; set [] for ALL 9 once it works
os.makedirs(CKPT_DIR, exist_ok=True)

df = pd.read_parquet(DATA_PATH)
if RESOLUTIONS:
    df = df[df.resolution.isin(RESOLUTIONS)].copy()
df["ts"] = pd.to_datetime(df["ts"])
df = df.sort_values(["series_id", "split", "ts"]).reset_index(drop=True)
def to_dict(split):
    return {sid: g.sort_values("ts")["value"].to_numpy(np.float32)
            for sid, g in df[df.split == split].groupby("series_id")}
TRAIN, VAL = to_dict("train"), to_dict("val")
META = df[["series_id","metric","resolution","vendor"]].drop_duplicates().set_index("series_id")
FULL = {sid: np.concatenate([TRAIN[sid], VAL.get(sid, np.array([], np.float32))]) for sid in TRAIN}
VAL_START = {sid: len(TRAIN[sid]) for sid in TRAIN}
print("series:", len(TRAIN))


In [ ]:
CONTEXT = 512
HORIZON = 24
EVAL_MAX_WINDOWS = 150
MODEL_ID = "amazon/chronos-2"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

def wape(y, p):
    y, p = np.asarray(y, float), np.asarray(p, float)
    d = np.abs(y).sum(); return float(np.abs(y-p).sum()/d*100) if d else float("nan")
def mase(y, p):
    y, p = np.asarray(y, float), np.asarray(p, float)
    mae = np.abs(y-p).mean(); naive = np.abs(np.diff(y)).mean() if len(y) > 1 else np.nan
    return float(mae/naive) if naive else float("nan")

# training inputs = full TRAIN series as list of {"target": array} (fit() windows internally)
train_inputs = [{"target": TRAIN[sid]} for sid in TRAIN if len(TRAIN[sid]) >= CONTEXT + HORIZON]
print("train_inputs:", len(train_inputs))

In [ ]:
from chronos import BaseChronosPipeline
pipe = BaseChronosPipeline.from_pretrained(MODEL_ID, device_map=DEVICE, torch_dtype=torch.bfloat16)
print("pipeline type:", type(pipe).__name__)
print("methods:", [m for m in dir(pipe) if not m.startswith("_")])
# quick predict smoke-test on one context (helps us confirm the call + output shape)
ctx = torch.tensor(FULL[list(FULL)[0]][:CONTEXT], dtype=torch.float32)
try:
    q, mean = pipe.predict_quantiles(ctx.unsqueeze(0), prediction_length=HORIZON,
                                     quantile_levels=[0.1, 0.5, 0.9])
    print("predict_quantiles OK | q shape:", tuple(q.shape), "| mean shape:", tuple(mean.shape))
except Exception as e:
    print("predict_quantiles failed:", type(e).__name__, e)


In [ ]:
_shape_printed = {"v": False}

@torch.no_grad()
def _median_forecast(pipe, ctx_batch, H):
    x = ctx_batch.unsqueeze(1).cpu()          # (B,1,C) CPU
    q, _ = pipe.predict_quantiles(x, prediction_length=H, quantile_levels=[0.5])
    def _np(a): return a.float().cpu().numpy() if hasattr(a, "cpu") else np.asarray(a, dtype=float)
    q = np.stack([_np(qi) for qi in q], 0) if isinstance(q, (list, tuple)) else _np(q)
    if not _shape_printed["v"]:
        print("stacked q shape:", q.shape, "| B:", x.shape[0]); _shape_printed["v"] = True
    B = x.shape[0]
    return np.asarray(q, dtype=float).reshape(B, -1)[:, :H]   # single quantile → flatten = H


@torch.no_grad()
def eval_holdout(pipe, full, val_start, meta, C, H, tag="model", batch=64,
                 max_windows=EVAL_MAX_WINDOWS):
    windows = []
    for sid, arr in full.items():
        vs = val_start[sid]
        origins = [o for o in range(vs, len(arr) - H + 1, H) if o - C >= 0]
        if max_windows and len(origins) > max_windows:
            origins = [origins[i] for i in np.linspace(0, len(origins)-1, max_windows).astype(int)]
        for o in origins:
            windows.append((sid, arr[o-C:o], arr[o:o+H]))
    per = {}
    for i in range(0, len(windows), batch):
        chunk = windows[i:i+batch]
        X = torch.tensor(np.stack([c for _, c, _ in chunk]), dtype=torch.float32, device=DEVICE)
        mp = _median_forecast(pipe, X, H)
        for j, (sid, _, tgt) in enumerate(chunk):
            per.setdefault(sid, ([], [])); per[sid][0].append(tgt); per[sid][1].append(mp[j])
    rows = []
    for sid, (ys, ps) in per.items():
        y = np.concatenate(ys); p = np.concatenate(ps); r = meta.loc[sid]
        rows.append(dict(series_id=sid, metric=r.metric, resolution=r.resolution,
                         vendor=r.vendor, tag=tag, wape=wape(y, p), mase=mase(y, p), n=len(y)))
    return pd.DataFrame(rows)

base_res = eval_holdout(pipe, FULL, VAL_START, META, CONTEXT, HORIZON, tag="zero_shot")
print("ZERO-SHOT  median WAPE:", round(base_res.wape.median(), 2),
      "| median MASE:", round(base_res.mase.median(), 2))
display(base_res.groupby("resolution")[["wape", "mase"]].median())


In [ ]:
!pip uninstall -y torchao

LORA_CONFIG = {
    "r": 8,
    "lora_alpha": 16,
    "target_modules": [
        "self_attention.q", "self_attention.v",
        "self_attention.k", "self_attention.o",
        "output_patch_embedding.output_layer",
    ],
}

ft = pipe.fit(
    train_inputs,
    prediction_length=HORIZON,
    finetune_mode="lora",
    learning_rate=1e-4,          # quickstart LoRA LR (1e-5 was too low)
    num_steps=3000,
    batch_size=32,
    context_length=CONTEXT,
    output_dir=CKPT_DIR,
    finetuned_ckpt_name="checkpoint",
    report_to="wandb",           # extra_trainer_kwargs → W&B curves
)
print("fine-tune done →", CKPT_DIR, "/checkpoint")


In [ ]:
ft_res = eval_holdout(ft, FULL, VAL_START, META, CONTEXT, HORIZON, tag="ft")
wandb.init(project=os.environ["WANDB_PROJECT"], name="holdout-compare")
wandb.log({"zero_shot_wape": float(base_res.wape.median()),
           "zero_shot_mase": float(base_res.mase.median()),
           "ft_wape": float(ft_res.wape.median()),
           "ft_mase": float(ft_res.mase.median())}); wandb.finish()

cmp = (pd.concat([base_res, ft_res]).groupby(["resolution", "tag"])[["wape","mase"]]
         .median().unstack())
display(cmp)
print("overall  zero-shot WAPE %.2f / MASE %.2f  →  ft WAPE %.2f / MASE %.2f" % (
    base_res.wape.median(), base_res.mase.median(), ft_res.wape.median(), ft_res.mase.median()))


In [ ]:
help(pipe.fit)                          # full signature + all parameters + defaults + docstring
import inspect
print(inspect.getsource(type(pipe).fit))   # full source of fit → see how it trains